# 00 · End-to-end workflow (one notebook that drives the .py modules)

Every step below is a function from `src/driftrev/`. The same sequence runs headless via
`python -m driftrev.pipeline --mode update` (cron / Docker scheduler).

| step | function | regime |
|---|---|---|
| 1 download (incremental) | `pipeline.ingest()` | `update` only |
| 2 transform → one table | `pipeline.transform()` / `pipeline.load_snapshot()` | `update` / `file` |
| 3 feature sets + split | `dataset.prepare()` | both |
| 4 train / tune / backtest | `pipeline.train_and_backtest()` | `--retrain` |
| 5 score newest events → signals | `pipeline.publish_live()` | both |
| 6 serve to agents / broker | `service.*`, `broker.preview_orders()` | both |


In [1]:
import sys, warnings, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
plt.rcParams.update({"figure.figsize": (11, 4.5), "axes.grid": True, "grid.alpha": 0.3})
from driftrev import config, store, artifacts, dataset, features, models, simulate, analysis, pipeline
print("data dir:", config.DATA_DIR)

data dir: /Users/mac/Documents/New project/stock-data-analysis/project/data


## 1 · Download (set `FETCH = True` to pull the latest data incrementally)

In [2]:
FETCH = False          # True: Yahoo + FRED + SEC EDGAR + Wikimedia, only the missing days
if FETCH:
    report = pipeline.ingest()
    print(json.dumps(report, indent=1, default=str)[:2000])
with store.connect() as con:
    sizes = {t: con.execute(f"select count(*) from {t}").fetchone()[0]
             for t in ["universe", "prices", "market", "macro", "filings", "eps_facts", "pageviews", "events", "daily_panel"]}
pd.Series(sizes, name="rows").to_frame()

,rows
universe,1502
prices,4426339
market,50226
macro,18348
filings,1896174
eps_facts,334462
pageviews,4522878
events,176802
daily_panel,4425338


## 2 · Transform (or load the snapshot in `file` regime)

In [3]:
TRANSFORM = False      # True: rebuild events + daily panel from the raw tables (~3 min)
data = pipeline.transform() if TRANSFORM else pipeline.load_from_warehouse()
events, panel = data["events"], data["panel"]
print(f"events: {len(events):,}   daily panel: {len(panel):,} rows   tickers: {panel.ticker.nunique():,}")
events[["date", "ticker", "event_family", "ret_1d", "edgar_category", "sue", "growth_future_10d"]].tail()

events: 176,802   daily panel: 4,425,338 rows   tickers: 1,485


,date,ticker,event_family,ret_1d,edgar_category,sue,growth_future_10d
176797,2026-11-13,STE,momentum_breakout,0.042929,offering,0.811537,NaN
176798,2026-11-13,VIR,shock_crash,-0.178218,offering,1.139782,NaN
176799,2026-11-13,VYX,momentum_breakout,0.052209,none,0.072033,NaN
176800,2026-11-13,WMT,momentum_breakout,0.015057,none,-0.061610,NaN
176801,2026-11-13,WWW,momentum_breakout,0.088325,none,0.992157,NaN


## 3 · Feature sets and temporal split

In [4]:
ml, feats = dataset.prepare(events)
print("TO_PREDICT:", dataset.TO_PREDICT)
print(f"NUMERIC: {len(dataset.numeric_features())}  DUMMIES from {dataset.CATEGORICAL}")
print(f"total model features: {len(feats)}")
ml.groupby("split").agg(rows=("ticker", "size"), start=("date", "min"), end=("date", "max"),
                        base_rate=(dataset.TARGET, "mean"))

TO_PREDICT: ['growth_future_5d', 'growth_future_10d', 'growth_future_20d', 'growth_future_5d_vs_spy', 'growth_future_10d_vs_spy', 'growth_future_20d_vs_spy', 'mae_10d', 'mfe_10d', 'is_positive_growth_10d', 'is_positive_growth_10d_vs_spy']
NUMERIC: 120  DUMMIES from ['event_family', 'edgar_category', 'sector', 'cap_bucket', 'month', 'weekday', 'regime_state', 'bin_reaction', 'bin_rsi', 'bin_sue', 'bin_52w', 'bin_vix', 'bin_attention', 'bin_momentum']
total model features: 202


,rows,start,end,base_rate
split,,,,
test,46831,2024-01-02,2026-11-13,0.399226
train,99735,2014-01-31,2021-12-31,0.361458
validation,30236,2022-01-03,2023-12-29,0.387650


## 4 · Train + backtest (`RETRAIN=True` refits everything; otherwise reuse the saved model)

In [5]:
RETRAIN = False
if RETRAIN:
    m = pipeline.train_and_backtest(ml, feats, panel, data["market"], tune=False)
metrics = artifacts.load_metrics()
bundle = models.load_bundle()
print("production model:", bundle["name"], "threshold:", bundle["threshold"],
      "val AUC %.3f  test AUC %.3f" % (bundle["val_auc"], bundle["test_auc"]))
pd.DataFrame(metrics["strategies"] + metrics["benchmarks"])[["name", "cagr", "sharpe", "max_drawdown", "n_trades"]]

production model: rf threshold: 0.47 val AUC 0.548  test AUC 0.541


,name,cagr,sharpe,max_drawdown,n_trades
0,all_events_unconditional,0.406138,1.028574,-0.495694,360.0
1,model_top5,0.352411,1.276343,-0.149241,92.0
2,model_top3_concentrated,0.147471,0.637883,-0.238199,58.0
3,model_top5_risk_managed,0.195153,1.390333,-0.109350,100.0
4,model_pead_only,0.091702,0.671276,-0.166269,52.0
5,model_crash_only,0.374832,1.579370,-0.097079,54.0
6,model_breakout_only,0.052808,0.358495,-0.255638,61.0
7,model_long_short,0.099424,0.654446,-0.194678,324.0
8,guide_armA_momentum_breakout,0.016433,0.249607,-0.077285,611.0
9,guide_armB_plus_consolidation,0.011424,0.296815,-0.055359,169.0


## 5 · Score the newest events → today's new trades

In [6]:
live = pipeline.publish_live(ml, panel, data["universe"], bundle)
print(live)
sig = artifacts.load_frame("signals")
recent = artifacts.load_frame("events")
cols = ["date", "ticker", "company", "event_family", "ret_1d", "prob", "tier", "expected_net_return",
        "entry_ref_price", "stop_loss_price", "take_profit_price", "exit_by_date", "edgar_category"]
print("latest session:", live["data_through"])
(sig if len(sig) else recent.head(15))[cols]

{'data_through': '2026-11-13', 'n_signals_today': 19, 'n_actionable_today': 1, 'n_events_90d': 3955}
latest session: 2026-11-13


,date,ticker,company,event_family,ret_1d,prob,tier,expected_net_return,entry_ref_price,stop_loss_price,take_profit_price,exit_by_date,edgar_category
0,2026-11-13,FUN,Six Flags,momentum_breakout,0.032172,0.471727,BUY,0.015917,11.550000,9.97,16.29,2026-11-26,none
1,2026-11-13,ARWR,Arrowhead Pharmaceuticals,earnings_up,0.013362,0.455422,WATCH,0.015917,47.779999,40.61,59.72,2026-11-30,earnings
2,2026-11-13,FLO,Flowers Foods,earnings_down,-0.079867,0.451151,WATCH,0.015917,5.530000,4.70,6.91,2026-11-30,earnings
3,2026-11-13,IBP,"Installed Building Products, Inc.",momentum_breakout,0.023418,0.449315,WATCH,0.015917,229.000000,208.04,291.87,2026-11-26,none
4,2026-11-13,AMAT,Applied Materials,earnings_down,-0.027431,0.448590,WATCH,0.015917,370.500000,314.92,463.12,2026-11-30,earnings
5,2026-11-13,CSCO,Cisco,earnings_down,-0.062445,0.444798,WATCH,0.015917,95.639999,81.29,119.55,2026-11-30,earnings
6,2026-11-13,VIR,"Vir Biotechnology, Inc.",shock_crash,-0.178218,0.441064,WATCH,0.015917,9.960000,8.47,12.45,2026-11-30,offering
7,2026-11-13,DIOD,Diodes Incorporated,earnings_down,-0.047489,0.439953,WATCH,0.015917,87.250000,74.16,109.06,2026-11-30,earnings
8,2026-11-13,ADMA,"ADMA Biologics, Inc.",momentum_breakout,0.033233,0.436358,WATCH,0.013481,13.680000,12.35,17.66,2026-11-26,none
9,2026-11-13,WWW,"Wolverine World Wide, Inc.",momentum_breakout,0.088325,0.431422,WATCH,0.013481,24.889999,22.63,31.66,2026-11-26,none


## 6 · What a 24/7 agent sees (same functions as the REST API / MCP server)

In [7]:
from driftrev import service, broker
print(service.latest_signals(limit=5)["summary"])
print(json.dumps(service.regime(), indent=1, default=str)[:800])
preview = broker.preview_orders(max_positions=5, capital=10_000)
pd.DataFrame(preview.get("orders", []))

Data through 2026-11-13; market regime: normal. 5 signal(s). Top: FUN (momentum_breakout) tier BUY, P(+2% in 10d)=47%, weight 10%, stop 9.97; ARWR (earnings_up) tier WATCH, P(+2% in 10d)=46%, weight 0%, stop 40.61; FLO (earnings_down) tier WATCH, P(+2% in 10d)=45%, weight 0%, stop 4.70; IBP (momentum_breakout) tier WATCH, P(+2% in 10d)=45%, weight 0%, stop 208.04; AMAT (earnings_down) tier WATCH, P(+2% in 10d)=45%, weight 0%, stop 314.92. Research output, not investment advice.
{
 "data_through": "2026-11-13",
 "regime": {
  "state": "normal",
  "points": 0,
  "as_of": "2026-11-13",
  "flags": {
   "flag_trend": 0,
   "flag_vix_term": 0,
   "flag_corr": 0,
   "flag_breadth": 0,
   "flag_credit": 0
  },
  "vix": 19.299999237060547,
  "vix_term": 0.9359844059918124,
  "breadth_above_200": 0.5569023569023569,
  "explain": "1 point per trigger: SPY<200d MA, VIX/VIX3M>1, sector-correlation pctile>0.9, breadth<40% above 200d MA, Baa-10y spread +25bp/20d. 0-1 normal, 2-3 elevated, 4+ stressed

,ticker,side,qty,tier,prob,weight,entry_ref_price,notional,stop_loss_price,take_profit_price,exit_by_date,order_type,time_in_force,order_class
0,FUN,buy,87,BUY,0.471727,0.101303,11.55,1004.85,9.97,16.29,2026-11-26,market,day,bracket
